# Manage flight data

Clean the dataset, build a simple model for delay risk, and export the airport list plus trained model.


In [ ]:
import csv
import pickle
from collections import defaultdict

source = Path('data/flights.csv')
airports_path = Path('data/airports.csv')
model_path = Path('data/model.pkl')

counts = defaultdict(int)
delays = defaultdict(int)
airports = {}
with source.open(newline='') as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        day = int(row.get('DayOfWeek') or 0)
        airport_id = int(row.get('DestAirportID') or 0)
        airport_name = (row.get('DestAirportName') or '').strip()
        if airport_id:
            airports[airport_id] = airport_name
        delay = int(row.get('ArrDel15') or 0)
        key = (day, airport_id)
        counts[key] += 1
        delays[key] += delay

model = {}
for (day, airport_id), total in counts.items():
    model[(day, airport_id)] = delays[(day, airport_id)] / total

with airports_path.open('w', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['id', 'name'])
    for airport_id in sorted(airports):
        writer.writerow([airport_id, airports[airport_id]])

with model_path.open('wb') as handle:
    pickle.dump(model, handle)

print('Generated', len(model), 'delay rates and', len(airports), 'airports.')
